
# Agent Öffentliche Ausschreibungen (ZUFALL logistics group)

### Aufbau:
  - TEIL 1  Konfiguration          Was gesucht wird (für alle Quellen gleich)
  - TEIL 2  Gemeinsames Format     Welche Spalten jede Quelle liefern muss
  - TEIL 3  Quellen (Adapter)      Pro Plattform ein Block: abrufen + umwandeln
  - TEIL 4  Prüfungen              Für alle Quellen gleich: CPV, Fristen Schlagwörter, Dubletten
  - TEIL 5  Ablauf (main)          Alles läuft hier zusammen

--> Neue Quelle hinzufügen: siehe Vorlage am Ende von TEIL 3.

In [ ]:
import io
import time
import zipfile
import xml.etree.ElementTree as ET
import requests
import pandas as pd

## TEIL 1: KONFIGURATION
- Alles, was ZUFALL oder ihr später anpassen wollt, steht hier und nur hier.

In [ ]:
SEIT_DATUM = "20260801"   # Bekanntmachungen ab diesem Datum (JJJJMMTT)
LAENDER = ["DEU"]         # Land des Auftraggebers

# NEU: CPV-Codes nach ZUFALL-Produkt gruppiert (aus ZUFALLs Kriterien-Excel)
PRODUKTE = {
    "Seefracht":        ["60600000", "60640000", "60650000"],
    # KLÄREN mit ZUFALL: 60440000 heißt laut ZUFALL "Luftfracht- und Posttransportdienste",
    # in den TED-Daten aber "Dienstleistungen unter Einsatz von Luftfahrzeugen".
    # Bringt Hagelabwehrflüge, Drohnen, Helikopter, Flugzieldarstellung, keine Luftfracht.
    # Ebenso 60410000 (Linienflugverkehr): bringt Passagierflüge ("PAX weltweit").
    "Luftfracht":       ["60400000", "60410000", "60420000", "60440000"],
    "Landverkehr":      ["60100000", "60180000"],
    "NSE":              ["60160000", "60161000", "64120000", "64121000", "64121200"],
    "Kontraktlogistik": ["63120000", "63121000", "63121100", "63122000"],
}

# Alle Codes als eine Liste, für Abfragen und Filter (wird aus PRODUKTE erzeugt)
CPV_CODES = [code for codes in PRODUKTE.values() for code in codes]

# NEU: Örtliche Eingrenzung. Nur Landverkehr ist regional begrenzt.
# ANNAHME, mit ZUFALL zu klären:
#   - Es zählt der LEISTUNGSORT (wo gefahren wird), nicht der Sitz des Auftraggebers
#   - Region = Landkreise der Standorte (NUTS-3), ohne Nachbarkreise
# NUTS-Codes am 03.11.2026 an echten Daten von öffentlichevergabe.de geprüft.
REGION_PRODUKTE = ["Landverkehr"]
REGION_KREISE = {
    "DE91C": "LK Göttingen",
    "DE731": "Kassel (Stadt)",
    "DE734": "LK Kassel",
    "DE732": "LK Fulda (Fulda, Eichenzell)",
    "DE722": "Lahn-Dill-Kreis (Haiger)",
    "DEG0G": "Weimarer Land (Grammetal)",
}

PERSONENVERKEHR = ["60112000", "60120000", "60130000", "60140000", "60170000"]

# NEU: von TED hierher verschoben, weil öffentlichevergabe.de dieselben Kürzel verwendet
NOTICE_TYPEN = [
    "cn-standard",       # Ausschreibung (auch Änderungsbekanntmachungen)
    "cn-social",         # Ausschreibung soziale Dienstleistungen
    "cn-desg",           # Planungswettbewerb
    "pin-cfc-standard",  # Vorinformation als Aufruf zum Wettbewerb
    "qu-sy",             # Qualifizierungssystem, mit Eike klären
]

AUSSCHLUSS_WOERTER = ["pax", "passagier", "fluggast", "personenbeförderung",
                      "schülerbeförderung", "schülerverkehr", "busverkehr", "linienverkehr"]

## TEIL 2: GEMEINSAMES FORMAT
- Der "Vertrag" zwischen den Quellen und dem Rest des Programms.
- Jede umwandeln-Funktion muss eine Tabelle mit genau diesen Spalten liefern.

In [ ]:
SPALTEN = [
    "Ausschreibungs ID Platform",  # ID auf der jeweiligen Plattform
    "Verfahrens ID",               # ID des Vergabeverfahrens, für Dubletten
    "Publikation",                 # Datum (datetime)
    "Art",                         # Art der Bekanntmachung
    "Titel",
    "Vergabe Branchen Code",       # Haupt-CPV-Code
    "Alle CPV",                    # alle CPV-Codes, kommagetrennt
    "Auftraggeber",
    "Stadt",
    "Leistungsort NUTS",           # NEU: Regionscodes des Leistungsorts, kommagetrennt
    "Angebotsfrist",               # Datum (datetime) oder leer
    "Fristart",                    # "Angebot", "Teilnahmeantrag" oder "keine Angabe"
    "Plattform",
    "Link zur Plattform",
]


def datum(text):
    """Macht aus '2026-09-08+02:00' oder '2026-12-01Z' ein echtes Datum. Leer bleibt leer."""
    if not text or pd.isna(text):
        return None
    return pd.to_datetime(str(text)[:10])


def erster(wert):
    """Gibt das erste Element zurück, wenn wert eine Liste ist."""
    if isinstance(wert, list) and len(wert) > 0:
        return wert[0]
    return wert


# NEU: aus TEIL 4 hierher verschoben, weil jetzt auch ein Adapter sie braucht
def cpv_passt(code, ziel):
    """CPV-Codes sind ein Baum: 60100000 umfasst alle Codes, die mit 601 beginnen."""
    return str(code).startswith(ziel.rstrip("0"))


def spalten_pruefen(tabelle, quelle):
    """Bricht mit klarer Meldung ab, wenn eine Quelle sich nicht an das Format hält."""
    fehlend = [s for s in SPALTEN if s not in tabelle.columns]
    if fehlend:
        raise ValueError(f"Quelle '{quelle}' liefert diese Spalten nicht: {fehlend}")
    return tabelle[SPALTEN]

## TEIL 3: QUELLEN (ADAPTER)
Jede Quelle hat zwei Funktionen:
1. ..._abrufen()        holt die Rohdaten, egal in welcher Form
2. ..._umwandeln(roh)   übersetzt sie in das gemeinsame Format aus TEIL 2

-> Eigenheiten einer Plattform bleiben in ihrem Block.

In [ ]:
TED_URL = "https://api.ted.europa.eu/v3/notices/search"

TED_FELDER = [
    "publication-number", "procedure-identifier", "publication-date", "notice-type",
    "notice-title", "main-classification-proc", "classification-cpv",
    "organisation-name-buyer", "organisation-city-buyer",
    "deadline-receipt-tender-date-lot", "deadline-receipt-request-date-lot",
    "place-of-performance",   # NEU: Leistungsort
]


def ted_anfrage_bauen():
    """TED kann schon serverseitig filtern. Das spart Datenmenge.
    Die CPV-Prüfung in TEIL 4 läuft trotzdem, damit alle Quellen gleich behandelt werden."""
    query = (
        f"classification-cpv IN ({' '.join(CPV_CODES)}) "
        f"AND buyer-country IN ({' '.join(LAENDER)}) "
        f"AND publication-date>={SEIT_DATUM} "
        f"AND NOT main-classification-proc IN ({' '.join(PERSONENVERKEHR)}) "
        f"AND notice-type IN ({' '.join(NOTICE_TYPEN)})"
    )
    return {
        "query": query,
        "fields": TED_FELDER,
        "limit": 250,
        "scope": "ACTIVE",
        "paginationMode": "ITERATION",   # Blätter-Modus
    }


def ted_abrufen():
    """Holt alle passenden Bekanntmachungen von TED als Liste."""
    anfrage = ted_anfrage_bauen()
    alle_notices = []

    while True:
        antwort = requests.post(TED_URL, json=anfrage, timeout=60)
        antwort.raise_for_status()   # bei Fehler abbrechen, main() fängt das ab

        daten = antwort.json()
        alle_notices = alle_notices + daten["notices"]

        # TED schickt auch nach der letzten Seite noch ein Lesezeichen
        # und fängt dann von vorne an. Deshalb auch bei leerer Seite abbrechen.
        token = daten.get("iterationNextToken")
        if not token or len(daten["notices"]) == 0:
            break
        anfrage["iterationNextToken"] = token

    return alle_notices


def ted_zeile(notice):
    """Übersetzt eine TED-Bekanntmachung in eine Zeile im gemeinsamen Format."""
    angebot = erster(notice.get("deadline-receipt-tender-date-lot"))
    teilnahme = erster(notice.get("deadline-receipt-request-date-lot"))

    if angebot:
        frist, fristart = angebot, "Angebot"
    elif teilnahme:
        frist, fristart = teilnahme, "Teilnahmeantrag"
    else:
        frist, fristart = None, "keine Angabe"

    # TED führt CPV-Codes pro Los auf, daher doppelte entfernen
    alle_cpv = sorted(set(notice.get("classification-cpv") or []))

    return {
        "Ausschreibungs ID Platform": notice.get("publication-number"),
        "Verfahrens ID": notice.get("procedure-identifier"),
        "Publikation": datum(notice.get("publication-date")),
        "Art": notice.get("notice-type"),
        "Titel": notice.get("notice-title", {}).get("deu"),
        "Vergabe Branchen Code": erster(notice.get("main-classification-proc")),
        "Alle CPV": ", ".join(alle_cpv),
        "Auftraggeber": erster(notice.get("organisation-name-buyer", {}).get("deu")),
        "Stadt": erster(notice.get("organisation-city-buyer")),
        "Leistungsort NUTS": ", ".join(sorted(set(notice.get("place-of-performance") or []))),   # NEU
        "Angebotsfrist": datum(frist),
        "Fristart": fristart,
        "Plattform": "TED",
        "Link zur Plattform": notice["links"]["html"]["DEU"],
    }


def ted_umwandeln(notices):
    return pd.DataFrame([ted_zeile(n) for n in notices], columns=SPALTEN)

In [ ]:
OEV_URL = "https://oeffentlichevergabe.de/api/notice-exports"


def oev_download(parameter):
    """Lädt ein ZIP von öffentlichevergabe.de. Bis zu 3 Versuche, die Seite bricht gelegentlich ab."""
    for versuch in range(3):
        try:
            antwort = requests.get(OEV_URL, params=parameter, timeout=180)
            antwort.raise_for_status()
            return zipfile.ZipFile(io.BytesIO(antwort.content))
        except requests.RequestException:
            if versuch == 2:
                raise
            time.sleep(10)


def oev_csv_lesen(zip_datei, name):
    """Liest eine der 19 CSV-Tabellen aus dem ZIP (z. B. 'notice', 'classification')."""
    datei = [d for d in zip_datei.namelist() if d.endswith(name + ".csv")][0]
    return pd.read_csv(zip_datei.open(datei), dtype=str)


def oev_treffer_aus_csv(zip_datei):
    """Führt die benötigten CSV-Tabellen zusammen und filtert grob vor.
    Ziel: möglichst wenige XML-Downloads im nächsten Schritt."""
    schluessel = ["noticeIdentifier", "noticeVersion"]   # verbindet alle Tabellen

    # Bekanntmachungen: nur die Arten, die wir wollen
    notice = oev_csv_lesen(zip_datei, "notice")
    notice = notice[notice["noticeType"].isin(NOTICE_TYPEN)]

    # CPV-Codes: Hauptcode + Zusatzcodes aller Lose einsammeln
    klass = oev_csv_lesen(zip_datei, "classification")
    klass = klass[klass["classificationType"] == "cpv"].copy()
    klass["codes"] = klass["mainClassificationCode"].fillna("") + "," + klass["additionalClassificationCodes"].fillna("")
    alle_cpv = (klass.groupby(schluessel)["codes"]
                .apply(lambda s: sorted({c.strip() for text in s for c in text.split(",") if c.strip()}))
                .rename("alle_cpv"))
    # Hauptcode = Zeile ohne Los (gilt fürs ganze Verfahren), sonst erste Zeile
    klass["ohne_los"] = klass["lotIdentifier"].isna()
    haupt_cpv = (klass.sort_values("ohne_los", ascending=False)
                 .groupby(schluessel)["mainClassificationCode"].first().rename("haupt_cpv"))

    # Auftraggeber: erste Organisation mit Rolle "buyer"
    org = oev_csv_lesen(zip_datei, "organisation")
    org = (org[org["organisationRole"] == "buyer"]
           .groupby(schluessel)[["organisationName", "organisationCity", "organisationCountryCode"]].first())

    # Titel: Zeile ohne Los = Titel des ganzen Verfahrens
    zweck = oev_csv_lesen(zip_datei, "purpose")
    titel = zweck[zweck["lotIdentifier"].isna()].groupby(schluessel)["title"].first()

    # NEU: Leistungsort, alle NUTS-Codes über alle Lose
    ort = oev_csv_lesen(zip_datei, "placeOfPerformance")
    nuts = (ort.dropna(subset=["placePerformanceCountrySubdivision"])
            .groupby(schluessel)["placePerformanceCountrySubdivision"]
            .apply(lambda s: ", ".join(sorted(set(s)))).rename("nuts"))

    treffer = (notice.set_index(schluessel)
               .join(alle_cpv).join(haupt_cpv).join(org).join(titel).join(nuts)
               .reset_index())

    # Grobfilter: Land, ZUFALL-CPV irgendwo, kein Personenverkehr als Hauptcode
    # (Bekanntmachungen ohne CPV-Code fallen hier raus, siehe offene Frage an ZUFALL)
    def passt(zeile):
        codes = zeile["alle_cpv"] if isinstance(zeile["alle_cpv"], list) else []
        if zeile["organisationCountryCode"] not in LAENDER:
            return False
        if any(cpv_passt(zeile["haupt_cpv"], p) for p in PERSONENVERKEHR):
            return False
        return any(cpv_passt(c, z) for c in codes for z in CPV_CODES)

    if treffer.empty:
        return treffer
    return treffer[treffer.apply(passt, axis=1)]


def oev_xml_auslesen(xml_inhalt):
    """Holt Frist und Link aus einer eForms-XML-Datei.
    {*} heißt: Namensraum egal. eForms verwendet mehrere Namensräume (cbc, cac, efac ...)."""
    wurzel = ET.fromstring(xml_inhalt)

    def fruehestes_datum(pfad):
        werte = [e.text for e in wurzel.findall(pfad) if e.text]
        return min(werte) if werte else None   # mehrere Lose -> früheste Frist zählt

    link = wurzel.find(".//{*}CallForTendersDocumentReference//{*}URI")
    return {
        "angebot": fruehestes_datum(".//{*}TenderSubmissionDeadlinePeriod/{*}EndDate"),
        "teilnahme": fruehestes_datum(".//{*}ParticipationRequestReceptionPeriod/{*}EndDate"),
        "link": link.text if link is not None else None,
    }


def oev_monate():
    """Alle Monate von SEIT_DATUM bis heute, z. B. ['2026-08', '2026-09', ...]."""
    return [p.strftime("%Y-%m") for p in pd.period_range(pd.Timestamp(SEIT_DATUM), pd.Timestamp.today(), freq="M")]


def oev_abrufen():
    """1. Monats-CSVs laden und filtern  2. Für Treffer-Tage das XML laden und Details ergänzen."""
    # Schritt 1: Treffer aus den Monats-CSVs
    teile = []
    for monat in oev_monate():
        teil = oev_treffer_aus_csv(oev_download({"pubMonth": monat, "format": "csv.zip"}))
        print(f"  öffentlichevergabe.de {monat}: {len(teil)} Treffer im CSV")
        teile.append(teil)
    treffer = pd.concat(teile, ignore_index=True)
    treffer["tag"] = treffer["publicationDate"].str[:10]
    treffer = treffer[treffer["tag"] >= pd.Timestamp(SEIT_DATUM).strftime("%Y-%m-%d")]

    # Schritt 2: Details aus dem XML, ein Download pro Veröffentlichungstag
    details = {}
    tage = sorted(treffer["tag"].unique())
    print(f"  öffentlichevergabe.de: lade XML für {len(tage)} Tage ...")
    for tag in tage:
        zip_datei = oev_download({"pubDay": tag, "format": "eforms.zip"})
        dateien = zip_datei.namelist()
        for nid in treffer.loc[treffer["tag"] == tag, "noticeIdentifier"]:
            passend = [d for d in dateien if d.startswith(nid)]
            if passend:
                details[nid] = oev_xml_auslesen(zip_datei.read(passend[-1]))

    treffer["details"] = treffer["noticeIdentifier"].map(details)
    return treffer


def oev_umwandeln(treffer):
    """Übersetzt die öffentlichevergabe.de-Treffer in das gemeinsame Format."""
    zeilen = []
    for _, t in treffer.iterrows():
        d = t["details"] if isinstance(t["details"], dict) else {}

        if d.get("angebot"):
            frist, fristart = d["angebot"], "Angebot"
        elif d.get("teilnahme"):
            frist, fristart = d["teilnahme"], "Teilnahmeantrag"
        else:
            frist, fristart = None, "keine Angabe"

        zeilen.append({
            "Ausschreibungs ID Platform": f"{t['noticeIdentifier']}-{t['noticeVersion']}",
            "Verfahrens ID": t["procedureIdentifier"],
            "Publikation": datum(t["publicationDate"]),
            "Art": t["noticeType"],
            "Titel": t["title"],
            "Vergabe Branchen Code": t["haupt_cpv"],
            "Alle CPV": ", ".join(t["alle_cpv"]) if isinstance(t["alle_cpv"], list) else "",
            "Auftraggeber": t["organisationName"],
            "Stadt": t["organisationCity"],
            "Leistungsort NUTS": t["nuts"] if isinstance(t["nuts"], str) else "",   # NEU
            "Angebotsfrist": datum(frist),
            "Fristart": fristart,
            "Plattform": "oeffentlichevergabe.de",
            "Link zur Plattform": d.get("link"),   # zeigt auf die Ursprungsplattform (dtvp, evergabe ...)
        })
    return pd.DataFrame(zeilen, columns=SPALTEN)

In [ ]:
QUELLEN = [
    {"name": "TED", "abrufen": ted_abrufen, "umwandeln": ted_umwandeln},
    {"name": "oeffentlichevergabe.de", "abrufen": oev_abrufen, "umwandeln": oev_umwandeln},
]

## TEIL 4: PRÜFUNGEN
- Gelten für alle Quellen gleich. Jede Prüfung gibt zwei Tabellen zurück: (behalten, aussortiert). Aussortiertes bekommt einen Grund, damit die Regeln später im Excel nachvollziehbar sind

In [ ]:
def produkte_ermitteln(codes):
    """Ordnet CPV-Codes ZUFALL-Produkten zu. Pro Code zählt der GENAUESTE passende
    ZUFALL-Code. Sonst fiele 60160000 (Post auf der Straße, NSE) auch unter
    60100000 (Straßentransport, Landverkehr), weil es im CPV-Baum darunter liegt."""
    produkte = set()
    for code in codes:
        kandidaten = [(len(ziel.rstrip("0")), produkt)
                      for produkt, ziele in PRODUKTE.items()
                      for ziel in ziele if cpv_passt(code, ziel)]
        if kandidaten:
            produkte.add(max(kandidaten)[1])   # längster Präfix = genauester Code
    return sorted(produkte)


def cpv_pruefen(tabelle):
    """Ordnet das ZUFALL-Produkt zu (NEU) und sortiert aus: kein ZUFALL-Code
    oder Personenverkehr als Hauptcode."""
    gruende, produkte = [], []
    for _, zeile in tabelle.iterrows():
        codes = [c.strip() for c in str(zeile["Alle CPV"]).split(",") if c.strip()]
        haupt = zeile["Vergabe Branchen Code"]
        gefunden = produkte_ermitteln(codes)
        produkte.append(", ".join(gefunden))

        if any(cpv_passt(haupt, p) for p in PERSONENVERKEHR):
            gruende.append("Personenverkehr (Hauptcode)")
        elif not gefunden:
            gruende.append("kein ZUFALL-CPV-Code")
        else:
            gruende.append(None)

    tabelle = tabelle.copy()
    tabelle["Zufall Produkt"] = produkte
    return aufteilen(tabelle, gruende)


def region_pruefen(tabelle):
    """NEU: Regional begrenzte Produkte (Landverkehr) nur im Gebiet der Standorte.
    Fliegt nur raus, wenn ALLE Produkte regional begrenzt sind. Landverkehr + Kontraktlogistik
    bleibt also drin, weil Kontraktlogistik keine örtliche Eingrenzung hat."""
    gruende, region = [], []
    for _, zeile in tabelle.iterrows():
        produkte = [p.strip() for p in zeile["Zufall Produkt"].split(",") if p.strip()]
        if not any(p in REGION_PRODUKTE for p in produkte):
            region.append("")
            gruende.append(None)
            continue

        nuts = [c.strip() for c in str(zeile["Leistungsort NUTS"]).split(",") if c.strip()]
        kreise = [c for c in nuts if len(c) == 5]          # NUTS-3 = Landkreis = 5 Zeichen
        im_gebiet = [REGION_KREISE[c] for c in kreise if c in REGION_KREISE]
        nur_regional = all(p in REGION_PRODUKTE for p in produkte)

        if im_gebiet:
            region.append("ja: " + ", ".join(im_gebiet))
            gruende.append(None)
        elif not kreise:
            region.append("unklar (kein Landkreis angegeben)")   # lieber drin lassen
            gruende.append(None)
        else:
            region.append("nein: " + ", ".join(kreise))
            gruende.append("Landverkehr außerhalb der Region" if nur_regional else None)

    tabelle = tabelle.copy()
    tabelle["Region Landverkehr"] = region
    return aufteilen(tabelle, gruende)


def nur_offene_fristen(tabelle):
    """Frist ab heute oder keine Frist angegeben."""
    heute = pd.Timestamp.today().normalize()
    abgelaufen = tabelle["Angebotsfrist"] < heute
    gruende = ["Frist abgelaufen" if a else None for a in abgelaufen]
    return aufteilen(tabelle, gruende)


def schlagwoerter_pruefen(tabelle):
    muster = r"\b(?:" + "|".join(AUSSCHLUSS_WOERTER) + ")"
    treffer = tabelle["Titel"].str.extract(f"({muster})", flags=2)[0]  # flags=2: Groß/klein egal
    gruende = [f"Schlagwort im Titel: {w.lower()}" if isinstance(w, str) else None for w in treffer]
    return aufteilen(tabelle, gruende)


def aufteilen(tabelle, gruende):
    """Hilfsfunktion: trennt Zeilen mit Grund (aussortiert) von Zeilen ohne (behalten)."""
    tabelle = tabelle.copy()
    tabelle["Ausschlussgrund"] = gruende
    raus = tabelle["Ausschlussgrund"].notna()
    behalten = tabelle[~raus].drop(columns="Ausschlussgrund")
    return behalten, tabelle[raus]


def dubletten_entfernen(tabelle):
    """Pro Verfahren nur die neueste Bekanntmachung behalten, auch über Quellen hinweg.
    Die Verfahrens-ID ist bei TED und öffentlichevergabe.de identisch (getestet 03.11.2026)."""
    tabelle = tabelle.copy()
    # Ohne Verfahrens-ID würden sonst alle leeren Zeilen als eine Dublette gelten
    tabelle["Verfahrens ID"] = tabelle["Verfahrens ID"].fillna(tabelle["Ausschreibungs ID Platform"])

    # NEU: festhalten, auf welchen Plattformen ein Verfahren gefunden wurde
    plattformen = tabelle.groupby("Verfahrens ID")["Plattform"].agg(lambda s: ", ".join(sorted(set(s))))
    tabelle["Gefunden auf"] = tabelle["Verfahrens ID"].map(plattformen)

    tabelle = tabelle.sort_values("Publikation")
    return tabelle.drop_duplicates(subset="Verfahrens ID", keep="last")


# Reihenfolge der Prüfungen. Neue Prüfung = Funktion schreiben + hier eintragen.
PRUEFUNGEN = [cpv_pruefen, region_pruefen, nur_offene_fristen, schlagwoerter_pruefen]

## TEIL 5: ABLAUF

In [ ]:
def alle_quellen_abrufen():
    """Ruft jede Quelle einzeln ab. Fällt eine aus, laufen die anderen weiter."""
    tabellen = []
    status = {}

    for quelle in QUELLEN:
        name = quelle["name"]
        print(f"Rufe {name} ab ...")
        try:
            roh = quelle["abrufen"]()
            tabelle = spalten_pruefen(quelle["umwandeln"](roh), name)
            tabellen.append(tabelle)
            status[name] = f"ok, {len(tabelle)} Bekanntmachungen"
        except Exception as fehler:
            status[name] = f"FEHLER: {fehler}"

    if not tabellen:
        return pd.DataFrame(columns=SPALTEN), status
    return pd.concat(tabellen, ignore_index=True), status


def main():
    # 1. Alle Quellen abrufen und zusammenführen
    tabelle, status = alle_quellen_abrufen()
    print()
    for name, text in status.items():
        print(f"{name}: {text}")

    # 2. Dubletten zuerst, damit danach die neueste Frist geprüft wird
    tabelle = dubletten_entfernen(tabelle)
    print("Nach Dubletten:", len(tabelle))

    # 3. Prüfungen der Reihe nach, Aussortiertes mit Grund sammeln
    aussortiert = []
    for pruefung in PRUEFUNGEN:
        tabelle, raus = pruefung(tabelle)
        aussortiert.append(raus)
        print(f"{pruefung.__name__}: {len(raus)} aussortiert, {len(tabelle)} übrig")

    aussortiert = pd.concat(aussortiert, ignore_index=True)
    relevant = tabelle

    print("\nRelevant:", len(relevant))
    print(relevant["Zufall Produkt"].value_counts().to_string())
    print("\nAussortiert nach Grund:")
    print(aussortiert["Ausschlussgrund"].value_counts().to_string())

    return relevant, aussortiert, status

relevant, aussortiert, status = main()

Rufe TED ab ...
Rufe oeffentlichevergabe.de ab ...

TED: ok, 93 Bekanntmachungen
oeffentlichevergabe.de: FEHLER: HTTPSConnectionPool(host='oeffentlichevergabe.de', port=443): Max retries exceeded with url: /api/notice-exports?pubMonth=2026-08&format=csv.zip (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ff2c6559810>, 'Connection to oeffentlichevergabe.de timed out. (connect timeout=180)'))
Nach Dubletten: 66
cpv_pruefen: 0 aussortiert, 66 übrig
region_pruefen: 16 aussortiert, 50 übrig
nur_offene_fristen: 29 aussortiert, 21 übrig
schlagwoerter_pruefen: 0 aussortiert, 21 übrig

Relevant: 21
Zufall Produkt
NSE                 11
Kontraktlogistik     6
Luftfracht           3
Seefracht            1

Aussortiert nach Grund:
Ausschlussgrund
Frist abgelaufen                    29
Landverkehr außerhalb der Region    16
